In [63]:
!pip install -U langchain langchain_openai langsmith pandas langchain_experimental matplotlib langgraph langchain_core duckduckgo-search langchain-community

In [64]:
import os
from uuid import uuid4

In [65]:
unique_id = uuid4().hex[:8]
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_PROJECT'] = f'Crypto Market Analysis AI Agent - Bithumb Leaderboard - {unique_id}'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] = ''
os.environ['OPENAI_API_KEY'] = ''

In [66]:
unique_id

'23eec22f'

1. 빗썸에 상장된 전체 종목 ticker list를 가져오는 함수 정의

In [67]:
import requests
import json

In [68]:
def get_bithumb_krw_ticker_list():
    url = 'https://api.bithumb.com/v1/market/all?isDetails=false'
    headers = {'accept': 'application/json'}
    response = requests.get(url, headers=headers)

    parsed_data = json.loads(response.text)
    krw_markets = [item for item in parsed_data if item['market'].startswith('KRW-')]
    market_list = [item['market'] for item in krw_markets]
    return market_list

In [69]:
market_list = get_bithumb_krw_ticker_list()
print(market_list)
print(len(market_list))

['KRW-BTC', 'KRW-ETH', 'KRW-ETC', 'KRW-XRP', 'KRW-BCH', 'KRW-QTUM', 'KRW-A', 'KRW-ICX', 'KRW-TRX', 'KRW-ELF', 'KRW-KNC', 'KRW-GLM', 'KRW-ZIL', 'KRW-WAXP', 'KRW-POWR', 'KRW-STEEM', 'KRW-ZRX', 'KRW-SNT', 'KRW-ADA', 'KRW-BAT', 'KRW-THETA', 'KRW-CVC', 'KRW-WAVES', 'KRW-LINK', 'KRW-ENJ', 'KRW-VET', 'KRW-MTL', 'KRW-IOST', 'KRW-AMO', 'KRW-BSV', 'KRW-ORBS', 'KRW-TFUEL', 'KRW-ANKR', 'KRW-CRO', 'KRW-CHR', 'KRW-MBL', 'KRW-FCT2', 'KRW-MEV', 'KRW-COS', 'KRW-EL', 'KRW-HIVE', 'KRW-XPR', 'KRW-EGG', 'KRW-BORA', 'KRW-ARPA', 'KRW-CTC', 'KRW-CKB', 'KRW-AERGO', 'KRW-UNI', 'KRW-YFI', 'KRW-UMA', 'KRW-AAVE', 'KRW-COMP', 'KRW-RSR', 'KRW-NMR', 'KRW-RLC', 'KRW-UOS', 'KRW-SAND', 'KRW-AWE', 'KRW-BEL', 'KRW-OBSR', 'KRW-POLA', 'KRW-ADP', 'KRW-GHX', 'KRW-CBK', 'KRW-MVC', 'KRW-GRT', 'KRW-BIOT', 'KRW-SNX', 'KRW-GRACY', 'KRW-OXT', 'KRW-MAPO', 'KRW-AQT', 'KRW-WIKEN', 'KRW-CTSI', 'KRW-MANA', 'KRW-LPT', 'KRW-SUSHI', 'KRW-PUNDIX', 'KRW-CELR', 'KRW-BFC', 'KRW-ALICE', 'KRW-OGN', 'KRW-COTI', 'KRW-CAKE', 'KRW-BNT', 'KRW-XVS', '

2. 특정 날짜의 빗썸 리더보드 순위 재현 함수

In [70]:
import time
from tqdm import tqdm
from datetime import datetime, timedelta

In [71]:
def get_bithumb_candle_data(market: str, to: str, count: int = 1):
    """
    Bithumb API에서 특정 시장과 날짜의 캔들 데이터를 가져오는 함수.
    
    Args:
        market (str): 조회할 마켓 (예: 'KRW_MVC').
        to (str): 특정 날짜의 마지막 캔들 시각 (ISO8061 포맷, 예: '2024-12-15T00:00:00').
        count (int, optional): 가져올 데이터 개수. 기본값은 1.
        
    Returns:
        dict: API 응답 데이터 (JSON 형식).
    """
    url = 'https://api.bithumb.com/v1/candles/days'

    params = {
        'count': count,
        'to' : to,
        'market' : market
    }

    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f'Error occurred: {e}')
        return None

In [72]:
def calculate_high_open_ratio_and_volume(result):
    """
    시가 대비 고가 상승률과 거래 대금을 억 단위로 계산하는 함수.

    Args:
        result (list): API 응답 데이터 (리스트 형식, 각 딕셔너리 내 시가, 고가, 거래 대금 포함).

    Returns:
        tuple: 시가 대비 고가 상승률(%)과 거래 대금(억 단위)을 반환.
    """
    if not result or 'opening_price' not in result[0] or 'high_price' not in result[0] or 'candle_acc_trade_price' not in result[0]:
        raise ValueError('유효한 데이터 형식이 아닙니다.')
    
    opening_price = result[0]['opening_price']
    high_price = result[0]['high_price']
    trade_volume = result[0]['candle_acc_trade_price']

    high_open_ratio = ((high_price - opening_price) / opening_price) *100
    trade_volume_in_100_million = trade_volume /1e8
    return high_open_ratio, trade_volume_in_100_million   

In [73]:
from langchain_core.tools import tool

In [74]:
@tool
def get_bithumb_daily_increase_rate_information(date_string: str) ->str:
    """
    주어진 날짜를 기준으로 빗썸의 모든 KRW 마켓의 시가 대비 고가 상승률과 거래 대급 정보를 계산하여 반환하는 함수.

    Parameters:
        date_string (str): 분석하고자 하는 기준 날짜. 형식은 "YYYY-MM-DD"이어야 합니다.

    Returns:
        str: 각 마켓의 시가 대비 고가 상승률과 거래 대금을 포함한 정렬된 결과 문자열.
        각 행은 "마켓명 시가 대비 고가 상승률: X.XX%, 거래 대금: Y.YY억 원" 형식으로 구성됩니다.

    주요 기능:
        1. 입력된 날짜 문자열을 기준으로 다음 날 자정(00:00:00)의 시간 문자열 생성.
        2. 빗썸의 KRW 마켓 리스트를 가져와 각 마켓의 데이터를 처리.
        3. 각 마켓의 시가 대비 고가 상승률 및 거래 대금 계산:
            -`calculate_high_open_ratio_and_volume`를 사용하여 상승률과 거래 대금 도출.
        4. 상승률(high_open_ratio)을 기준으로 결과 정렬(내림차순).
        5. 최종적으로 각 마켓의 결과를 문자열로 포맷하여 변환.
    
    참고:
        - `get_bithumb_krw_ticker_list`: 빗썸의 KRW 마켓 리스트를 가져오는 함수.
        - `get_bithumb_candle_data`: 특정 마켓과 날짜에 해당하는 캔들 데이터를 가져오는 함수.
        - `calculate_high_open_ratio_and_volume`: 상승률 및 거래 대금을 계산하는 함수.
        - `time.sleep(0.01)`: API 호출 간격 조정.
        - 결과는 시가 대비 고가 상승률 기준으로 내림차순 정렬됩니다.

    예외 처리:
        - 데이터가 유효하지 않을 경우 해당 마켓은 건너뜀.
        - `calculate_high_open_ratio_and_volume` 처리 중 오류 발생 시 해당 마켓 건너뜀.
    """
    market_list = get_bithumb_krw_ticker_list()
    current_date = datetime.strptime(date_string, "%Y-%m-%d")
    results = []

    """
    날짜 문자열 생성
    """
    next_date = current_date + timedelta(days=1)
    date_str = next_date.strftime('%Y-%m-%dT00:00:00')

    """
    시장 데이터 처리 루프
    """
    for market in market_list:
        to = date_str

        """
        데이터 가져오기
        """
        result = get_bithumb_candle_data(market, to)

        if result:
            try:
                """
                상승률과 거래 대금 계산
                """
                ratio, volume = calculate_high_open_ratio_and_volume(result)
                
                """
                결과 저장
                """
                results.append({
                    'market': market,
                    'high_open_ratio': ratio,
                    'trade_volume': volume
                })
            except ValueError as e:
                print(f"{market} 처리 중 에러: {e}")

        """
        요청 간격 조정
        """
        time.sleep(0.01)
    current_date += timedelta(days=1)
    """
    시가 대비 고가 상승률 기준으로 결과 정렬
    """
    sorted_results = sorted(results, key=lambda x: x['high_open_ratio'], reverse=True)

    """
    정렬된 결과 출력
    """
    output = '\n'.join(
        f"{item['market']} 시가 대비 고가 상승률: {item['high_open_ratio']:.2f}%, 거래 대금: {item['trade_volume']:.2f}억 원"
        for item in sorted_results
    )
    return output

In [75]:
print(get_bithumb_daily_increase_rate_information.name)
print(get_bithumb_daily_increase_rate_information.description)
print(get_bithumb_daily_increase_rate_information.args)

get_bithumb_daily_increase_rate_information
주어진 날짜를 기준으로 빗썸의 모든 KRW 마켓의 시가 대비 고가 상승률과 거래 대급 정보를 계산하여 반환하는 함수.

Parameters:
    date_string (str): 분석하고자 하는 기준 날짜. 형식은 "YYYY-MM-DD"이어야 합니다.

Returns:
    str: 각 마켓의 시가 대비 고가 상승률과 거래 대금을 포함한 정렬된 결과 문자열.
    각 행은 "마켓명 시가 대비 고가 상승률: X.XX%, 거래 대금: Y.YY억 원" 형식으로 구성됩니다.

주요 기능:
    1. 입력된 날짜 문자열을 기준으로 다음 날 자정(00:00:00)의 시간 문자열 생성.
    2. 빗썸의 KRW 마켓 리스트를 가져와 각 마켓의 데이터를 처리.
    3. 각 마켓의 시가 대비 고가 상승률 및 거래 대금 계산:
        -`calculate_high_open_ratio_and_volume`를 사용하여 상승률과 거래 대금 도출.
    4. 상승률(high_open_ratio)을 기준으로 결과 정렬(내림차순).
    5. 최종적으로 각 마켓의 결과를 문자열로 포맷하여 변환.

참고:
    - `get_bithumb_krw_ticker_list`: 빗썸의 KRW 마켓 리스트를 가져오는 함수.
    - `get_bithumb_candle_data`: 특정 마켓과 날짜에 해당하는 캔들 데이터를 가져오는 함수.
    - `calculate_high_open_ratio_and_volume`: 상승률 및 거래 대금을 계산하는 함수.
    - `time.sleep(0.01)`: API 호출 간격 조정.
    - 결과는 시가 대비 고가 상승률 기준으로 내림차순 정렬됩니다.

예외 처리:
    - 데이터가 유효하지 않을 경우 해당 마켓은 건너뜀.
    - `calculate_high_open_ratio_and_volume` 처리 중 오류 발생 시 해당 마켓 건너뜀.
{

In [76]:
coin_data = get_bithumb_daily_increase_rate_information.invoke({'date_string':'2025-01-01'})
print(coin_data)

KRW-CVC 시가 대비 고가 상승률: 21.36%, 거래 대금: 270.72억 원
KRW-AERGO 시가 대비 고가 상승률: 17.11%, 거래 대금: 126.98억 원
KRW-STEEM 시가 대비 고가 상승률: 16.88%, 거래 대금: 225.22억 원
KRW-GLM 시가 대비 고가 상승률: 16.33%, 거래 대금: 81.75억 원
KRW-TEMCO 시가 대비 고가 상승률: 16.33%, 거래 대금: 49.45억 원
KRW-VIRTUAL 시가 대비 고가 상승률: 15.24%, 거래 대금: 304.68억 원
KRW-NFT 시가 대비 고가 상승률: 14.29%, 거래 대금: 0.56억 원
KRW-XLM 시가 대비 고가 상승률: 13.02%, 거래 대금: 277.28억 원
KRW-ACS 시가 대비 고가 상승률: 12.33%, 거래 대금: 22.35억 원
KRW-LM 시가 대비 고가 상승률: 11.13%, 거래 대금: 27.10억 원
KRW-PENGU 시가 대비 고가 상승률: 11.09%, 거래 대금: 238.80억 원
KRW-GAS 시가 대비 고가 상승률: 10.21%, 거래 대금: 3.75억 원
KRW-ROA 시가 대비 고가 상승률: 9.98%, 거래 대금: 34.80억 원
KRW-TDROP 시가 대비 고가 상승률: 9.70%, 거래 대금: 12.31억 원
KRW-HUNT 시가 대비 고가 상승률: 9.56%, 거래 대금: 13.56억 원
KRW-EGG 시가 대비 고가 상승률: 9.42%, 거래 대금: 9.32억 원
KRW-OBSR 시가 대비 고가 상승률: 8.89%, 거래 대금: 10.63억 원
KRW-CTC 시가 대비 고가 상승률: 8.50%, 거래 대금: 73.04억 원
KRW-FLR 시가 대비 고가 상승률: 8.44%, 거래 대금: 10.85억 원
KRW-HIVE 시가 대비 고가 상승률: 8.07%, 거래 대금: 143.55억 원
KRW-CARV 시가 대비 고가 상승률: 7.90%, 거래 대금: 31.99억 원
KRW-CRTS 시가 대비 고가 상승률:

crypto research agent

In [77]:
tools = [get_bithumb_daily_increase_rate_information]

In [78]:
from langchain_openai import ChatOpenAI

In [79]:
llm = ChatOpenAI(
    model = 'gpt-4o-mini',
    temperature=0,
)

In [80]:
llm_with_tools = llm.bind_tools(tools)

In [81]:
from langchain_core.prompts import ChatPromptTemplate

In [82]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """당신은 LangSmith 에이전트입니다.
사용자의 질문에 답변하고 필요한 경우 도구를 사용합니다."""
    ),
    ("human", "{input}")
])

print(prompt.pretty_repr())

================================ System Message ================================

당신은 LangSmith 에이전트입니다.
사용자의 질문에 답변하고 필요한 경우 도구를 사용합니다.

================================ Human Message =================================

{input}


In [84]:
from langchain_classic.agents import AgentExecutor
from langchain_classic.agents.format_scratchpad.openai_tools import(
    format_to_openai_tool_messages,
)
from langchain_classic.agents.output_parsers.openai_tools import OpenAIToolsAgentOutputParser

In [ ]:
runnable_agent = (
    {
        'input': lambda x:x['inpout']
        
    }
)